# Running a simple demo task using v6's python client

In [ ]:
from getpass import getpass
from vantage6.client import Client
import json
from datetime import date

In [ ]:
config = {
    # MODIFY:
    "server_url": "http://127.0.6.1",
    #"server_url": "https://example.medicaldataworks.nl",
    # MODIFY:
    "server_port": 80,
    #"server_port": 443,
    "server_api": "/api",
    # MODIFY:
    # your user name goes here
    "username": "phobos",
    #"username": "",
    # Your password goes here
    "password": "test-password-two-orbit",
    # Or, better: ask for it:
    #"password": getpass("Password: "),
    # MODIFY:
    # You can generate this private key in your node by running
    # `v6 node create-private-key`
    # Then copy that file to your local machine and point here to it
    #"organization_key": "/some/path/to/your/private-key.pem",
    "organization_key": None,
}

In [ ]:
# Initialize the client object, and run the authentication & encryption setup
client = Client(config["server_url"], config["server_port"], config["server_api"])
client.authenticate(config["username"], config["password"])
client.setup_encryption(config["organization_key"])

## Asking for partials from each node directly

In [ ]:
# MODIFY:
target_org_ids = [1, 2, 3]
# MODIFY:
target_collab_id = 1
# MODIFY:
task_image = "ghcr.io/mdw-nl/v6-average-py:v1.0.1"
task_name = "Average"
# MODIFY:
databases = [
    {"label": "letters"}
]
# MODIFY:
input_ = {
    "method": "partial_average",
    "kwargs": {
        "column_name": "value",
    }
}


print(f"Running task {target_org_ids}")
task = client.task.create(
    collaboration=target_collab_id,
    organizations=target_org_ids,
    name=f"{date.today():%Y-%m-%d} | {task_name}",
    image=task_image,
    description="",
    databases=databases,
    input_=input_,
)

print(f"Waiting for results of task ID: {task["id"]}")
client.wait_for_results(task["id"])
print(f"\nResults are in.")
print()

In [ ]:
result_info = client.result.from_task(task_id=task["id"])

for result in result_info.get("data"):
    run = client.run.get(result["run"]["id"])
    org_id = run["organization"]["id"]
    actual_result = result["result"]
    org_name = client.organization.get(org_id)["name"]
    print(f"Results from organization {org_name} (ID: {org_id}):")
    # we load it and dump it again to pretty-print the JSON
    try:
        print(json.dumps(json.loads(actual_result), indent=2))
    except (json.JSONDecodeError, TypeError) as e:
        print("Error decoding JSON, printing raw result:")
        print(actual_result)
    print()

## Asking aggregator to ask for partials and aggregate them

In [ ]:
# MODIFY: (aggregator)
target_org_ids = [1]
# MODIFY:
target_org_ids_partial = [1, 2, 3]
# MODIFY:
target_collab_id = 1
# MODIFY:
task_image = "ghcr.io/mdw-nl/v6-average-py:v1.0.1"
task_name = "Average"
# MODIFY:
databases = [
    {"label": "letters"}
]
# MODIFY:
input_ = {
    "method": "central_average",
    "kwargs": {
        "column_name": "value",
        "org_ids": target_org_ids_partial,
    }
}


print(f"Running task {target_org_ids}")
task = client.task.create(
    collaboration=target_collab_id,
    organizations=target_org_ids,
    name=f"{date.today():%Y-%m-%d} | {task_name}",
    image=task_image,
    description="",
    databases=databases,
    input_=input_,
)

print(f"Waiting for results of task ID: {task["id"]}")
client.wait_for_results(task["id"])
print(f"\nResults are in.")
print()

In [ ]:
result_info = client.result.from_task(task_id=task["id"])

for result in result_info.get("data"):
    run = client.run.get(result["run"]["id"])
    org_id = run["organization"]["id"]
    actual_result = result["result"]
    org_name = client.organization.get(org_id)["name"]
    print(f"Results from organization {org_name} (ID: {org_id}):")
    # we load it and dump it again to pretty-print the JSON
    try:
        print(json.dumps(json.loads(actual_result), indent=2))
    except (json.JSONDecodeError, TypeError) as e:
        print("Error decoding JSON, printing raw result:")
        print(actual_result)
    print()